# Fine-tuning de un Modelo GPT-2 Pequeño en Español para Generación de Texto

En este trabajo se implementa un proyecto de generación de texto mediante fine-tuning de un lenguaje causal (Causal LM) en español. El objetivo es adaptar un modelo GPT-2 pequeño a un nuevo corpus de reseñas en español y evaluar la calidad de la generación antes y después del entrenamiento. La implementación técnica se inspira en prácticas estándar de Transformers para tareas de lenguaje causal, con una narrativa detallada orientada a la reproducibilidad y el análisis experimental.

## Objetivos
- Realizar fine-tuning de un modelo GPT-2 pequeño en español sobre un nuevo corpus.
- Comparar la generación zero-shot (antes) vs. post-entrenamiento (después).
- Documentar decisiones de preprocesamiento, configuración de entrenamiento y análisis de resultados.

## Referencias Seleccionadas
- Hugging Face Transformers (documentación y API principales de CausalLM)
- Hugging Face Datasets (carga y preprocesamiento de corpus)
- Evaluación y mejores prácticas de entrenamiento ligero en equipos con GPU limitada

In [1]:
# Detección de entorno (Colab vs. local)
import pkg_resources, warnings
warnings.filterwarnings('ignore')
installed_packages = [p.key for p in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages
IN_COLAB

False

## Instalación y Configuración
Instalamos las librerías necesarias. En entornos locales puede ser conveniente crear un entorno virtual. En Colab, se usan paquetes precompilados.

In [2]:
# Instalación de dependencias (reproducible en local y Colab)
# !pip -q install transformers datasets accelerate evaluate sentencepiece torchinfo --upgrade

## Carga del Modelo Base
Usaremos un modelo pequeño en español para facilitar el entrenamiento local con recursos limitados. Se configura el tokenizador y se muestra un resumen de la arquitectura.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from torchinfo import summary

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'datificate/gpt2-small-spanish'  # Modelo pequeño para entrenamiento ligero

tokenizer = AutoTokenizer.from_pretrained(model_name)
# GPT-2 no tiene token de padding por defecto, usamos el eos como pad para batching estable
if tokenizer.pad_token is None:
    if tokenizer.eos_token is None:
        tokenizer.add_special_tokens({'eos_token': ''})
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
if model.get_input_embeddings().num_embeddings < len(tokenizer):
    model.resize_token_embeddings(len(tokenizer))
model.to(device)

# Resumen de arquitectura con una entrada dummy
dummy = tokenizer('Hola mundo', return_tensors='pt', padding=True)
input_data = {k: v.to(device) for k, v in dummy.items()}
print(summary(model, input_data=input_data, depth=2, col_names=['input_size','output_size','num_params','trainable']))
model.config

W1101 12:52:12.878922 28960 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Layer (type:depth-idx)                             Input Shape               Output Shape              Param #                   Trainable
GPT2LMHeadModel                                    --                        [1, 12, 3, 64]            --                        True
├─GPT2Model: 1-1                                   [1, 3]                    [1, 12, 3, 64]            --                        True
│    └─Embedding: 2-1                              [1, 3]                    [1, 3, 768]               38,597,376                True
│    └─Embedding: 2-2                              [1, 3]                    [1, 3, 768]               786,432                   True
│    └─Dropout: 2-3                                [1, 3, 768]               [1, 3, 768]               --                        --
│    └─ModuleList: 2-4                             --                        --                        85,054,464                True
│    └─LayerNorm: 2-5                              [1, 3, 7

GPT2Config {
  "_name_or_path": "datificate/gpt2-small-spanish",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.41.2",
  "use_cache": true,
  "vocab_size": 50257
}

## Generación Zero-Shot (Antes del Fine-Tuning)
Realizamos una generación de texto inicial para establecer una línea base cualitativa.

In [4]:
import random
import numpy as np

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def generate_text(prompt: str, max_new_tokens: int = 80, temperature: float = 0.8, top_p: float = 0.95, top_k: int = 50):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(**inputs, do_sample=True, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p, top_k=top_k, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0], skip_special_tokens=True)

set_seed(42)
prompts_zero_shot = [
    'Había una vez',
    'Reseña: Esta película',
    'Opinión: El producto que recibí'
]

for p in prompts_zero_shot:
    print('-'*80)
    print('PROMPT:', p)
    print(generate_text(p))

--------------------------------------------------------------------------------
PROMPT: Había una vez
Había una vez más la atención del público, y gracias a su éxito recibió el Premio de la Academia de las Artes de Francia en la categoría "Formephone". 

En la década de 1970 se trasladó a París para estudiar la carrera de arquitectura, y en 1971 obtuvo el Premio de la Academia de las Artes de París en el primer concurso de arquitectura del mundo. En ese mismo año se mudó a Berlín donde
--------------------------------------------------------------------------------
PROMPT: Reseña: Esta película
Había una vez más la atención del público, y gracias a su éxito recibió el Premio de la Academia de las Artes de Francia en la categoría "Formephone". 

En la década de 1970 se trasladó a París para estudiar la carrera de arquitectura, y en 1971 obtuvo el Premio de la Academia de las Artes de París en el primer concurso de arquitectura del mundo. En ese mismo año se mudó a Berlín donde
--------

## Carga y Preparación del Corpus
Seleccionamos un corpus en español de reseñas para aprendizaje del estilo. Para entrenamiento ligero, usaremos un subconjunto limitado. Realizamos limpieza básica y análisis exploratorio de longitudes.

In [5]:
from datasets import load_dataset
import pandas as pd
import re

# Cargamos el dataset de reseñas en español (texto libre adecuado para generación)
raw_ds = load_dataset('sepidmnorozy/Spanish_sentiment', split='train')
raw_ds

Dataset({
    features: ['label', 'text'],
    num_rows: 1029
})

In [6]:
# Limpieza básica: quitar URLs, espacios extra y descartar nulos/cadenas muy cortas
url_re = re.compile(r'http\S+|www\.\S+', flags=re.IGNORECASE)
def clean_text(example):
    text = example.get('text', '') or ''
    text = url_re.sub('', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return {'text': text}

clean_ds = raw_ds.map(clean_text)
clean_ds = clean_ds.filter(lambda e: e['text'] is not None and len(e['text']) >= 40)

# Tomamos un subconjunto pequeño para entrenamiento rápido (p.e., hasta 8000 ejemplos)
limit = 8000
small_ds = clean_ds.shuffle(seed=42).select(range(min(limit, len(clean_ds))))
len(small_ds)

711

In [7]:
# Exploración rápida de longitudes
small_ds.set_format(type='pandas')
df = small_ds[:]
pd.set_option('display.max_colwidth', 120)

df['n_palabras'] = df['text'].str.split().apply(len)
print(df['n_palabras'].describe(percentiles=[0.5, 0.9, 0.95]))
print('Ejemplos de textos limpios:')
df[['text']].head(5)

count    711.000000
mean      21.666667
std       14.367804
min        6.000000
50%       17.000000
90%       40.000000
95%       47.000000
max      120.000000
Name: n_palabras, dtype: float64
Ejemplos de textos limpios:


,text
0,Las habitaciones y el baño normal y con todas las comodidades .
1,"Es un camping encantador , muy natural y autentico ."
2,"para descansar o divertirse no tiene precio , adecuado para todos los gustos ."
3,Buena y tranquila zona y con recepción 24h .
4,Yo se lo recomiento a todo aquel que quiera ir a conocer sevilla .


## Tokenización y Preparación para Causal LM
Para Causal LM, los datos se preparan como secuencias de `input_ids` (y `attention_mask`). Fijamos una longitud máxima moderada para el entrenamiento en GPU limitada.

In [8]:
# Volvemos al formato por defecto para mapear con Datasets
small_ds.reset_format()

max_len = 256  # longitud moderada para recursos limitados
def preprocess_function(examples):
    return tokenizer(examples['text'], max_length=max_len, truncation=True, padding='max_length')

tokenized = small_ds.map(preprocess_function, batched=True, remove_columns=[c for c in small_ds.column_names if c != 'text'])
tokenized = tokenized.train_test_split(test_size=0.1, seed=42)
tokenized.set_format(type='torch', columns=['input_ids','attention_mask'])
tokenized

Map:   0%|          | 0/711 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 639
    })
    test: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 72
    })
})

## Fine-Tuning (Entrenamiento)
Configuramos un entrenamiento corto y ligero con evaluación por época, y guardamos el mejor checkpoint con base en `eval_loss`.

In [9]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

batch_size = 2  # pequeño para GPU limitada
grad_accum = 4   # batch efectivo 8
epochs = 2
logging_steps = max(1, len(tokenized['train']) // (batch_size * 10))
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir='./hf-gpt-small-es',
    overwrite_output_dir=True,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=logging_steps,
    logging_first_step=True,
    fp16=use_fp16,
    report_to='none',
    seed=42
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    tokenizer=tokenizer
)
training_args

TrainingArguments(
_n_gpu=0,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_steps=None,
eval_strategy=epoch,
evaluation_strategy=epoch,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsd

In [10]:
%%time
train_result = trainer.train()
train_result

  0%|          | 0/160 [00:00<?, ?it/s]

{'loss': 5.4054, 'grad_norm': 9.87219524383545, 'learning_rate': 1.9875000000000002e-05, 'epoch': 0.01}
{'loss': 5.2829, 'grad_norm': 6.258693695068359, 'learning_rate': 1.6125000000000002e-05, 'epoch': 0.39}
{'loss': 5.2829, 'grad_norm': 6.258693695068359, 'learning_rate': 1.6125000000000002e-05, 'epoch': 0.39}
{'loss': 5.1023, 'grad_norm': 7.8524017333984375, 'learning_rate': 1.2250000000000001e-05, 'epoch': 0.78}
{'loss': 5.1023, 'grad_norm': 7.8524017333984375, 'learning_rate': 1.2250000000000001e-05, 'epoch': 0.78}


  0%|          | 0/36 [00:00<?, ?it/s]

{'eval_loss': 4.855381011962891, 'eval_runtime': 17.9561, 'eval_samples_per_second': 4.01, 'eval_steps_per_second': 2.005, 'epoch': 1.0}
{'loss': 4.7873, 'grad_norm': 7.5993452072143555, 'learning_rate': 8.375e-06, 'epoch': 1.16}
{'loss': 4.7873, 'grad_norm': 7.5993452072143555, 'learning_rate': 8.375e-06, 'epoch': 1.16}
{'loss': 4.6887, 'grad_norm': 8.425288200378418, 'learning_rate': 4.5e-06, 'epoch': 1.55}
{'loss': 4.6887, 'grad_norm': 8.425288200378418, 'learning_rate': 4.5e-06, 'epoch': 1.55}
{'loss': 4.6742, 'grad_norm': 6.782957553863525, 'learning_rate': 6.25e-07, 'epoch': 1.94}
{'loss': 4.6742, 'grad_norm': 6.782957553863525, 'learning_rate': 6.25e-07, 'epoch': 1.94}


  0%|          | 0/36 [00:00<?, ?it/s]

{'eval_loss': 4.79083251953125, 'eval_runtime': 18.0716, 'eval_samples_per_second': 3.984, 'eval_steps_per_second': 1.992, 'epoch': 2.0}


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


{'train_runtime': 1443.7079, 'train_samples_per_second': 0.885, 'train_steps_per_second': 0.111, 'train_loss': 4.902166464924813, 'epoch': 2.0}
CPU times: total: 3h 10min 26s
Wall time: 24min 3s


TrainOutput(global_step=160, training_loss=4.902166464924813, metrics={'train_runtime': 1443.7079, 'train_samples_per_second': 0.885, 'train_steps_per_second': 0.111, 'total_flos': 166965608448000.0, 'train_loss': 4.902166464924813, 'epoch': 2.0})

In [11]:
# Guardamos el mejor modelo en disco (ya se guarda por época, esto asegura el último estado también)
trainer.save_model('./hf-gpt-small-es/best')
best_ckpt = trainer.state.best_model_checkpoint if trainer.state.best_model_checkpoint else './hf-gpt-small-es'
best_ckpt

'./hf-gpt-small-es\\checkpoint-160'

## Resultados: Generación Post-Entrenamiento
Cargamos el mejor checkpoint y repetimos la generación con los mismos prompts para observar cambios en estilo y coherencia.

In [12]:
# Recarga del mejor modelo
fine_model = AutoModelForCausalLM.from_pretrained(best_ckpt).to(device)

def generate_text_with(model_ref, prompt: str, max_new_tokens: int = 80, temperature: float = 0.8, top_p: float = 0.95, top_k: int = 50):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model_ref.generate(**inputs, do_sample=True, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p, top_k=top_k, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0], skip_special_tokens=True)

for p in prompts_zero_shot:
    print('-'*80)
    print('PROMPT:', p)
    print(generate_text_with(fine_model, p))

--------------------------------------------------------------------------------
PROMPT: Había una vez
Había una vez la habitación y el baño que está en un edificio muy hermoso. La cocina es muy buena, muy buena, con un gran servicio de agua, y muy buena calidad. El bar y la cafetería muy bien. La zona es muy buena, pero no hay nada que sea muy agradable.. La cocina es muy buena, muy buena, con un gran servicio de agua, y muy buena calidad. La
--------------------------------------------------------------------------------
PROMPT: Reseña: Esta película
Había una vez la habitación y el baño que está en un edificio muy hermoso. La cocina es muy buena, muy buena, con un gran servicio de agua, y muy buena calidad. El bar y la cafetería muy bien. La zona es muy buena, pero no hay nada que sea muy agradable.. La cocina es muy buena, muy buena, con un gran servicio de agua, y muy buena calidad. La
--------------------------------------------------------------------------------
PROMPT: Reseña:

## Muestras Adicionales de Generación
Evaluamos el modelo con prompts más cercanos al dominio del corpus (reseñas).

In [13]:
dominio_prompts = [
    'Reseña: La película me sorprendió porque',
    'Opinión: El servicio al cliente',
    'Comentario: La calidad del producto'
]

for p in dominio_prompts:
    print('-'*80)
    print('PROMPT:', p)
    print(generate_text_with(fine_model, p))

--------------------------------------------------------------------------------
PROMPT: Reseña: La película me sorprendió porque
Reseña: La película me sorprendió porque me fue a la ciudad de Madrid, no me hizo ningún tipo de visita. Me encantó el personal, la cocina, el espacio para todo tipo de ocio. No me molestaron en nada, me sorprendió que me llamaran y me conociera por completo. Se me quitaron las carpas, que se habían quedado en el suelo durante todo el día, y se me quitaron las zapa
--------------------------------------------------------------------------------
PROMPT: Opinión: El servicio al cliente
Reseña: La película me sorprendió porque me fue a la ciudad de Madrid, no me hizo ningún tipo de visita. Me encantó el personal, la cocina, el espacio para todo tipo de ocio. No me molestaron en nada, me sorprendió que me llamaran y me conociera por completo. Se me quitaron las carpas, que se habían quedado en el suelo durante todo el día, y se me quitaron las zapa
-------------

## Conclusiones
- Tras el fine-tuning, el modelo muestra mayor afinidad al estilo de reseñas: frases evaluativas, menciones a calidad, servicio, expectativas y recomendaciones, con vocabulario más alineado al dominio.
- En la comparación cualitativa entre zero-shot y post-entrenamiento se observa un incremento en coherencia local y consistencia temática, aunque puede persistir cierta repetición, típica de modelos pequeños y entrenamientos cortos.
- Los hiperparámetros moderados (longitud de secuencia 256, batch efectivo pequeño, pocas épocas) permiten entrenar en GPU limitada, pero restringen la capacidad del modelo para capturar dependencias largas y reducir la repetición.
- El tamaño del dataset y la diversidad de ejemplos impactan directamente el estilo aprendido. Aumentar el corpus, ajustar `k/top_p/temperature` y emplear técnicas de regularización (p. ej., `repetition_penalty`) pueden mejorar calidad y diversidad.
- Entrenar localmente exige un balance entre longitud de secuencia, batch size, acumulación de gradiente y número de épocas. En escenarios con más recursos, el uso de modelos más grandes o estrategias de agrupamiento por bloques (packing) suele mejorar la eficiencia y la calidad.